# Long-Tailed Object Detection: Yolo11 baseline

This notebook anchors the baseline training pipeline for drone-view object detection using YOLOv11 **without any pretrained weights**.
Fill in each `# TODO` placeholder before executing the training cell below.


In [ ]:
from pathlib import Path
from datetime import datetime
from typing import List, Tuple, Optional
import random
import shutil

from PIL import Image
import yaml

PROJECT_DIR = Path.cwd()
DATA_ROOT = (PROJECT_DIR / '../../../dataset/taica-cvpdl-2025-hw-2/CVPDL_hw2/CVPDL_hw2').resolve()
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f'Dataset train folder not found at {TRAIN_DIR}. Update DATA_ROOT if your layout differs.')

if not TEST_DIR.exists():
    print(f'Warning: test folder not found at {TEST_DIR}. Test images will not be copied.')
    TEST_DIR = None

EXPERIMENT_NAME = 'yolov11_baseline_from_scratch'
RUN_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
EXPERIMENT_DIR = (PROJECT_DIR / 'artifacts' / EXPERIMENT_NAME / RUN_ID).resolve()
YOLO_DATA_DIR = EXPERIMENT_DIR / 'dataset'
RUN_ROOT_DIR = EXPERIMENT_DIR / 'run'
TRAIN_RUN_NAME = 'train'
VAL_RUN_NAME = 'val'
INFER_RUN_NAME = 'infer'

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT_DIR.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.2  # fixed 80/20 train/val split
SEED = 11


def load_label_file(label_path: Path) -> List[Tuple[int, float, float, float, float]]:
    boxes: List[Tuple[int, float, float, float, float]] = []
    with label_path.open('r') as fh:
        for raw_line in fh:
            line = raw_line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(',')]
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:5])
            boxes.append((cls, x, y, w, h))
    return boxes


def convert_tlwh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int) -> Tuple[float, float, float, float]:
    xc = (x + w / 2.0) / img_w
    yc = (y + h / 2.0) / img_h
    ww = w / img_w
    hh = h / img_h
    xc = min(max(xc, 0.0), 1.0)
    yc = min(max(yc, 0.0), 1.0)
    ww = min(max(ww, 0.0), 1.0)
    hh = min(max(hh, 0.0), 1.0)
    return xc, yc, ww, hh


def prepare_dataset(train_dir: Path, output_dir: Path, val_ratio: float, seed: int, test_dir: Optional[Path] = None):
    if output_dir.exists():
        shutil.rmtree(output_dir)
    images_root = output_dir / 'images'
    labels_root = output_dir / 'labels'
    for split in ['train', 'val']:
        (images_root / split).mkdir(parents=True, exist_ok=True)
        (labels_root / split).mkdir(parents=True, exist_ok=True)
    if test_dir is not None:
        (images_root / 'test').mkdir(parents=True, exist_ok=True)
        (labels_root / 'test').mkdir(parents=True, exist_ok=True)

    image_stems = sorted({path.stem for path in train_dir.glob('*.png')})
    missing_labels: List[str] = []
    missing_images: List[str] = []

    for label_path in train_dir.glob('*.txt'):
        if not (train_dir / f'{label_path.stem}.png').exists():
            missing_images.append(label_path.stem)

    records: List[Tuple[str, Path, Path]] = []
    for stem in image_stems:
        image_path = train_dir / f'{stem}.png'
        label_path = train_dir / f'{stem}.txt'
        if not label_path.exists():
            missing_labels.append(stem)
            continue
        records.append((stem, image_path, label_path))

    rng = random.Random(seed)
    rng.shuffle(records)

    if not records:
        raise RuntimeError('No image/label pairs found in the training dataset.')

    if len(records) == 1:
        val_count = 0
    else:
        val_count = max(1, int(len(records) * val_ratio))
        val_count = min(len(records) - 1, val_count)

    val_stems = {stem for stem, *_ in records[:val_count]}

    stats = {
        'train': 0,
        'val': 0,
        'test': 0,
        'boxes': 0,
        'missing_labels': missing_labels,
        'missing_images': missing_images,
        'class_ids': set(),
    }

    for stem, image_path, label_path in records:
        split = 'val' if stem in val_stems else 'train'
        with Image.open(image_path) as img:
            img_w, img_h = img.size

        label_entries = load_label_file(label_path)
        yolo_lines = []
        for cls, x, y, w, h in label_entries:
            xc, yc, ww, hh = convert_tlwh_to_yolo(x, y, w, h, img_w, img_h)
            yolo_lines.append(f'{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}')
            stats['class_ids'].add(cls)

        dst_img = images_root / split / f'{stem}.png'
        shutil.copy2(image_path, dst_img)

        dst_label = labels_root / split / f'{stem}.txt'
        dst_label.write_text('\n'.join(yolo_lines))

        stats[split] += 1
        stats['boxes'] += len(yolo_lines)

    if test_dir is not None:
        test_images = sorted(test_dir.glob('*.png'))
        for image_path in test_images:
            shutil.copy2(image_path, images_root / 'test' / image_path.name)
            stats['test'] += 1

    stats['val_ratio'] = val_ratio
    return stats


stats = prepare_dataset(
    TRAIN_DIR,
    YOLO_DATA_DIR,
    val_ratio=VAL_RATIO,
    seed=SEED,
    test_dir=TEST_DIR,
)

print(f'Dataset prepared at {YOLO_DATA_DIR}')
print(f"train images: {stats['train']} | val images: {stats['val']} | boxes: {stats['boxes']}")
if stats['test']:
    print(f"test images copied: {stats['test']}")
if stats['missing_labels']:
    sample = ', '.join(stats['missing_labels'][:5])
    print(f"Skipped {len(stats['missing_labels'])} images without labels. Examples: {sample}")
if stats['missing_images']:
    sample = ', '.join(stats['missing_images'][:5])
    print(f"Skipped {len(stats['missing_images'])} labels without images. Examples: {sample}")

class_ids = sorted(stats['class_ids'])
if not class_ids:
    raise RuntimeError('No class ids found in annotations. Check label parsing logic.')

ALL_CLASS_NAMES = {
    0: 'car',
    1: 'hov',
    2: 'person',
    3: 'motorcycle',
}
missing_class_ids = sorted(set(class_ids) - set(ALL_CLASS_NAMES))
if missing_class_ids:
    raise ValueError(f'Unknown class ids {missing_class_ids} detected. Update ALL_CLASS_NAMES mapping to include them.')

CLASS_NAME_MAP = {cid: ALL_CLASS_NAMES[cid] for cid in class_ids}
DATA_CONFIG_PATH = EXPERIMENT_DIR / 'longtail_dataset.yaml'

data_yaml = {
    'path': YOLO_DATA_DIR.as_posix(),
    'train': 'images/train',
    'val': 'images/val',
}
if stats['test']:
    data_yaml['test'] = 'images/test'
data_yaml['nc'] = len(CLASS_NAME_MAP)
data_yaml['names'] = {cid: name for cid, name in CLASS_NAME_MAP.items()}

with DATA_CONFIG_PATH.open('w') as fh:
    yaml.safe_dump(data_yaml, fh, sort_keys=False)

print(f'YOLO data config saved to {DATA_CONFIG_PATH}')




In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from ultralytics import YOLO


@dataclass
class TrainConfig:
    """Configuration container for baseline YOLOv11 training."""

    # TODO: choose the YOLOv11 architecture (Ultralytics bundle names such as 'yolov11n.yaml')
    model_yaml: str = 'yolo11s.yaml'

    # Dataset YAML generated in the preparation cell above
    dataset_yaml: Path = DATA_CONFIG_PATH

    # TODO: adjust batch size based on GPU memory
    batch_size: int = 16

    # TODO: adjust epoch count based on convergence needs
    epochs: int = 100

    image_size: int = 640

    # Use CUDA if available; override with a specific device string if needed (e.g. "0", "0,1", "cpu")
    device: str = 'auto'

    workers: int = 8
    project_dir: Path = RUN_ROOT_DIR
    run_name: str = TRAIN_RUN_NAME
    val_name: str = VAL_RUN_NAME
    seed: int = SEED

    # Optimization hyperparameters for a plain baseline run
    optimizer: str = 'SGD'
    learning_rate: float = 0.01
    final_lr_ratio: float = 0.01  # ratio between final and initial LR (Ultralytics uses cosine by default)
    weight_decay: float = 5e-4
    momentum: float = 0.937
    warmup_epochs: float = 3.0

    # Early stopping patience in epochs; adjust if you need longer training
    patience: int = 50

    # Optional: resume from an earlier run (leave as None for a fresh start)
    resume_checkpoint: Optional[Path] = None


def set_deterministic(seed: int) -> None:
    """Set random seeds for reproducible training."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def validate_paths(config: TrainConfig) -> None:
    """Ensure critical input files exist before launching training."""

    if '#TODO' in config.model_yaml:
        raise ValueError('Update TrainConfig.model_yaml with the YOLOv11 architecture name or YAML path you intend to use.')

    dataset_yaml_path = Path(config.dataset_yaml)
    if not dataset_yaml_path.exists():
        raise FileNotFoundError(f'Dataset YAML not found at: {dataset_yaml_path}')

    if config.resume_checkpoint is not None:
        resume_path = Path(config.resume_checkpoint)
        if not resume_path.exists():
            raise FileNotFoundError(f'Resume checkpoint not found at: {resume_path}')


In [ ]:
from pathlib import Path
from typing import Any, Dict


def build_model(config: TrainConfig) -> YOLO:
    """Instantiate a YOLOv11 model without loading pretrained weights."""
    model = YOLO(config.model_yaml)
    model.overrides['pretrained'] = False
    return model


def build_train_kwargs(config: TrainConfig) -> Dict[str, Any]:
    """Generate the argument dictionary passed into `YOLO.train`."""
    return {
        'data': str(config.dataset_yaml),
        'epochs': config.epochs,
        'batch': config.batch_size,
        'imgsz': config.image_size,
        'device': config.device,
        'workers': config.workers,
        'project': str(config.project_dir),
        'name': config.run_name,
        'exist_ok': True,
        'optimizer': config.optimizer,
        'lr0': config.learning_rate,
        'lrf': config.final_lr_ratio,
        'weight_decay': config.weight_decay,
        'momentum': config.momentum,
        'warmup_epochs': config.warmup_epochs,
        'patience': config.patience,
        'resume': str(config.resume_checkpoint) if config.resume_checkpoint else False,
        'pretrained': False,
        'save_period': -1,  # only keep best/last checkpoints
        'verbose': True,
    }


config = TrainConfig()

set_deterministic(config.seed)
validate_paths(config)
config.project_dir.mkdir(parents=True, exist_ok=True)

if config.device == 'auto':
    resolved_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Auto-selected device: {resolved_device}")
    config.device = resolved_device

model = build_model(config)
train_kwargs = build_train_kwargs(config)
train_result = model.train(**train_kwargs)

run_dir = Path(getattr(train_result, 'save_dir', config.project_dir / config.run_name))
print(f'Training artifacts saved to: {run_dir}')

best_weights_path = run_dir / 'weights' / 'best.pt'
if best_weights_path.exists():
    print(f'Best checkpoint available at: {best_weights_path}')
else:
    print(f'Warning: best checkpoint not found at {best_weights_path}')

TRAINED_CONFIG = config
TRAINED_MODEL = model
TRAIN_RUN_DIR = run_dir
BEST_WEIGHTS_PATH = best_weights_path

train_result


In [ ]:
val_results = model.val(
    data=str(config.dataset_yaml),
    imgsz=config.image_size,
    batch=config.batch_size,
    device=config.device,
    workers=config.workers,
    split='val',
    save_json=False,
    plots=True,
    verbose=True,
    project=str(TRAINED_CONFIG.project_dir),
    name=TRAINED_CONFIG.val_name,
    exist_ok=True,
)

VAL_RUN_DIR = Path(val_results.save_dir)
print(f"Validation metrics saved under: {VAL_RUN_DIR}")
val_results


In [ ]:
import csv
from pathlib import Path

if TEST_DIR is None:
    raise RuntimeError('TEST_DIR is None; run the preparation cell with a valid test set before inference.')

test_images_dir = YOLO_DATA_DIR / 'images/test'
if not test_images_dir.exists():
    raise FileNotFoundError(f'Test images folder not found at {test_images_dir}. Prepare the dataset again before running inference.')

best_weights = BEST_WEIGHTS_PATH if 'BEST_WEIGHTS_PATH' in globals() else TRAIN_RUN_DIR / 'weights' / 'best.pt'
if not best_weights.exists():
    raise FileNotFoundError(f'Best checkpoint not found at {best_weights}. Train the model before running inference.')

inference_model = YOLO(str(best_weights))
predict_results = inference_model.predict(
    source=str(test_images_dir),
    project=str(TRAINED_CONFIG.project_dir),
    name=INFER_RUN_NAME,
    imgsz=TRAINED_CONFIG.image_size,
    device=TRAINED_CONFIG.device,
    conf=0.01,
    save=True,
    save_txt=True,
    save_conf=True,
    exist_ok=True,
)

if not predict_results:
    raise RuntimeError('No predictions were returned by YOLO inference.')

prediction_save_dir = Path(predict_results[0].save_dir)
print(f'Inference outputs saved to: {prediction_save_dir}')

submission_rows = []
for result in predict_results:
    stem = Path(result.path).stem
    digits = ''.join(ch for ch in stem if ch.isdigit())
    image_id = str(int(digits)) if digits else stem
    image_h, image_w = result.orig_shape
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        pred_string = ''
    else:
        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        classes = boxes.cls.cpu().numpy()
        parts = []
        for conf, (x1, y1, x2, y2), cls in zip(confs, xyxy, classes):
            x1 = max(0.0, float(x1))
            y1 = max(0.0, float(y1))
            x2 = min(float(x2), float(image_w))
            y2 = min(float(y2), float(image_h))
            width = max(0.0, x2 - x1)
            height = max(0.0, y2 - y1)
            parts.append(f'{conf:.6f} {x1:.2f} {y1:.2f} {width:.2f} {height:.2f} {int(cls)}')
        pred_string = ' '.join(parts)
    sort_key = int(image_id) if image_id.isdigit() else image_id
    submission_rows.append((sort_key, image_id, pred_string))

submission_rows.sort(key=lambda item: item[0])

submission_path = EXPERIMENT_DIR / 'submission.csv'
with submission_path.open('w', newline='') as fh:
    writer = csv.writer(fh)
    writer.writerow(['Image_ID', 'PredictionString'])
    for _, image_id, prediction in submission_rows:
        writer.writerow([image_id, prediction])

print(f'Submission file written to: {submission_path}')
SUBMISSION_PATH = submission_path
INFERENCE_RESULTS = predict_results
PREDICTION_SAVE_DIR = prediction_save_dir
submission_path
